#Experimentos

## Importacion de librerias

In [ ]:
import os, sys, json, copy, warnings, itertools
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn
import joblib
import matplotlib
matplotlib.use("Agg")          # sin pantalla en Colab
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
)
from transformers import (
    Wav2Vec2Processor, Wav2Vec2Model,
    Wav2Vec2ForSequenceClassification,
)

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuracion del entorno

Directorios de trabajo

In [ ]:
WORKSPACE      = "/content/drive/My Drive/disfluencias/"
SYNTHETIC_ROOT = WORKSPACE + "sintetic_speakers/"
SYNTHETIC_CSV  = WORKSPACE + "disfluencias_general.csv"
REAL_CSV       = WORKSPACE + "anotaciones.csv"
REAL_CLIPS_CSV = WORKSPACE + "clips_info.csv"

Parametros de inicio, y preparacion de entorno de pytorch

In [ ]:
TARGET_SR    = 16_000
N_MFCC       = 40
WAV2VEC_ID   = "facebook/wav2vec2-base-960h"
RANDOM_STATE = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Declaracion de tipos de disfluencia para ambas fuentes

In [ ]:
# Columnas de disfluencia del dataset real (FluencyBank)
REAL_DIS_COLS  = ["Prolongation", "Block", "SoundRep", "WordRep", "Interjection"]
# Tipos disponibles en el CSV sintético
SYNTH_TIPOS    = ["rep_pal", "prol", "bloq", "rep_sil"]
SYNTH_SEV      = ["leve", "moderado", "severo"]
SYNTH_INTEN    = ["baja", "media", "alta"]

In [ ]:
OUTPUT_DIR  = Path(WORKSPACE) / "pipeline_outputs_v3"
MODELS_DIR  = OUTPUT_DIR / "models"
FIGS_DIR    = OUTPUT_DIR / "figures"
CACHE_DIR   = OUTPUT_DIR / "feature_cache"
for d in [MODELS_DIR, FIGS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
print(f"[Config] Dispositivo: {DEVICE}")
print(f"[Config] Outputs en: {OUTPUT_DIR}")

[Config] Dispositivo: cuda
[Config] Outputs en: /content/drive/My Drive/disfluencias/pipeline_outputs_v3


# Carga de datos
Selector: source="real" | source="synthetic" | source="both"

## Carga de datos reales
Se realiza el siguiente preprocesamiento:

1.   Leer dataset original con los peakers
2.   Filtrar show HELR, PoorAudioQuality=0, DifficultToUnderstand=0
3. Crear columna is_disfluent (binaria)
4. Determinar tipo dominante de disfluencia
5. Merge con rutas de audio (clips_info.csv)





Columnas de salida (mínimo):
      wav_audio_path, is_disfluent,
disfluency_type,
      speaker, source='real', intensidad=None, severidad=None

In [ ]:
def load_real_data() -> pd.DataFrame:
    ann = pd.read_csv(REAL_CSV)

    # Preprocesamiento de calidad
    n_raw = len(ann)
    ann = ann[ann["Show"] != "HELR"]
    ann = ann[
        (ann["PoorAudioQuality"] == 0) &
        (ann["DifficultToUnderstand"] == 0)
    ].copy()
    print(f"    Filtrado calidad: {n_raw} → {len(ann)} clips")

    # Etiqueta binaria
    ann["is_disfluent"] = ann[REAL_DIS_COLS].any(axis=1).astype(int)

    # Tipo dominante (primera columna activa)
    def _dom(row):
        for c in REAL_DIS_COLS:
            if row[c]:
                return c
        return "fluent"
    ann["disfluency_type"] = ann.apply(_dom, axis=1)

    # Merge con rutas de audio
    clips = pd.read_csv(REAL_CLIPS_CSV)
    df = pd.merge(ann, clips, on=["Show", "EpId", "ClipId"], how="left")

    # Columnas de trazabilidad
    df["speaker"]    = df["Show"]
    df["source"]     = "real"
    df["intensidad"] = None
    df["severidad"]  = None

    missing = df["wav_audio_path"].isnull().sum()
    if missing:
        print(f"    ⚠ {missing} clips sin ruta de audio")
    df = df.dropna(subset=["wav_audio_path"]).reset_index(drop=True)

    n_dis = df["is_disfluent"].sum()
    print(f"    Total: {len(df)} clips | Disfluentes: {n_dis} ({n_dis/len(df):.1%})")
    print(f"    Speakers: {df['speaker'].nunique()}")
    return df

## Carga de datos sinteticos

Lee el archivo JSON de alineación de un speaker.
Estructura esperada del JSON (una entrada por chunk):
```
    [
      {
        "start": 0.0,
        "end":   2.94,
        "text":  "A menos de ocho meses …",
        "audio": "chunk_001.wav"     ← nombre del clip dentro de output_aligned/<folder>/
      },
      ...
    ]
```
Devuelve lista de dicts con campos: start, end, text, audio.

Devuelve [] si el archivo no existe o no es parseable.


In [ ]:
def _parse_alignment_json(json_path: str) -> list[dict]:
    p = Path(json_path)
    if not p.exists():
        return []
    try:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        # Aceptar tanto lista como dict con key "chunks"
        if isinstance(data, dict):
            data = data.get("chunks", data.get("segments", []))
        return data if isinstance(data, list) else []
    except Exception as e:
        print(f"    [JSON] Error leyendo {json_path}: {e}")
        return []

  Recorre la estructura de carpetas de datos sintéticos y construye un DataFrame con una fila por clip de audio (.wav de 3s).
  

```
 Estructura de carpetas:
      synth_root/
        <speaker>/
          output_aligned/
            <audio_name>/        ← un folder por audio original
              clip_001.wav
              clip_002.wav
              …
          work/
            alignment/
              <audio_name>.json  ← chunks con start/end/text
            <audio_name>.wav     ← audio original completo
```

Proceso:

1. Leer CSV de disfluencias sintéticas y aplicar filtros
2. Para cada speaker → carpeta output_aligned
3. Para cada sub-carpeta (=audio original):
  . Buscar JSON de alineación en work/alignment/
           b. Cruzar clips .wav con metadata del CSV (por nombre de archivo)
           c. Si hay JSON: enriquecer con timestamps start/end/text
4. Clips sin match en CSV → is_disfluent=0 (fluido)


In [ ]:

def discover_synthetic_clips(
    synth_root: str         = SYNTHETIC_ROOT,
    synth_csv:  str         = SYNTHETIC_CSV,
    intensidad_filter: list = None,
    severidad_filter:  list = None,
    tipo_filter:       list = None,
    speaker_filter:    list = None,
) -> pd.DataFrame:

    # Leer y filtrar CSV
    ann = pd.read_csv(synth_csv)
    if intensidad_filter:
        ann = ann[ann["intensidad"].isin(intensidad_filter)]
    if severidad_filter:
        ann = ann[ann["severidad"].isin(severidad_filter)]
    if tipo_filter:
        ann = ann[ann["tipo_disfluencia"].isin(tipo_filter)]

    # Índice por stem de archivo para búsqueda rápida
    csv_idx = defaultdict(list)
    for _, row in ann.iterrows():
        csv_idx[Path(row["archivo"]).stem].append(row.to_dict())

    records = []
    root    = Path(synth_root)

    for sp_dir in sorted(root.iterdir()):
        if not sp_dir.is_dir():
            continue
        speaker = sp_dir.name
        if speaker_filter and speaker not in speaker_filter:
            continue

        out_aligned = sp_dir / "output_aligned"
        alignment_dir = sp_dir / "work" / "alignment"

        if not out_aligned.exists():
            continue

        for audio_folder in sorted(out_aligned.iterdir()):
            if not audio_folder.is_dir():
                continue

            folder_key = audio_folder.name   # mismo stem que el .txt del CSV

            # Metadata del CSV para este audio
            csv_rows = csv_idx.get(folder_key, [])

            # Cargar JSON de alineación si existe
            json_path  = alignment_dir / f"{folder_key}.json"
            align_data = _parse_alignment_json(str(json_path))

            # Índice de alineación por nombre de clip
            align_by_clip = {}
            for chunk in align_data:
                clip_name = Path(chunk.get("audio", "")).name
                if clip_name:
                    align_by_clip[clip_name] = chunk

            # Determinar metadata para todos los clips de esta carpeta
            if csv_rows:
                # Si hay varias filas (distintas disfluencias en mismo audio)
                # usar la primera como representativa para is_disfluent/tipo
                base_meta = {
                    "is_disfluent":    1,
                    "disfluency_type": csv_rows[0]["tipo_disfluencia"],
                    "intensidad":      csv_rows[0]["intensidad"],
                    "severidad":       csv_rows[0]["severidad"],
                    "num_disfluencias": csv_rows[0].get("num_disfluencias", 1),
                }
            else:
                # No aparece en CSV → fluido
                base_meta = {
                    "is_disfluent":    0,
                    "disfluency_type": "fluent",
                    "intensidad":      None,
                    "severidad":       None,
                    "num_disfluencias": 0,
                }

            # Añadir un registro por cada clip .wav
            for clip_path in sorted(audio_folder.glob("*.wav")):
                clip_name = clip_path.name
                ainfo = align_by_clip.get(clip_name, {})
                records.append({
                    "wav_audio_path":  str(clip_path),
                    "is_disfluent":    base_meta["is_disfluent"],
                    "disfluency_type": base_meta["disfluency_type"],
                    "intensidad":      base_meta["intensidad"],
                    "severidad":       base_meta["severidad"],
                    "num_disfluencias":base_meta["num_disfluencias"],
                    "speaker":         speaker,
                    "source":          "synthetic",
                    "audio_folder":    folder_key,
                    # Datos de alineación (vacíos si no hay JSON)
                    "chunk_start":     ainfo.get("start"),
                    "chunk_end":       ainfo.get("end"),
                    "chunk_text":      ainfo.get("text", ""),
                })

    df = pd.DataFrame(records)
    if df.empty:
        print("    ⚠ No se encontraron clips. Verifica SYNTHETIC_ROOT.")
        return df

    n_dis = df["is_disfluent"].sum()
    print(f"    Total clips: {len(df)} | Disfluentes: {n_dis} ({n_dis/len(df):.1%})")
    print(f"    Speakers   : {df['speaker'].nunique()}")
    print(f"    Tipos      : {df['disfluency_type'].value_counts().to_dict()}")
    return df

## Selector de fuente de datos

    Punto de entrada unificado para carga de datos.

    Args:
        source       : "real"      → solo datos reales
                       "synthetic" → solo datos sintéticos
                       "both"      → concat real + sintético
        synth_kwargs : dict de filtros para discover_synthetic_clips
                       (intensidad_filter, severidad_filter, tipo_filter,
                        speaker_filter)

    Returns:
        DataFrame con columnas comunes:
          wav_audio_path, is_disfluent, disfluency_type,
          speaker, source, intensidad, severidad

In [ ]:

def load_data(
    source: str = "both",
    synth_kwargs: dict = None,
) -> pd.DataFrame:
    synth_kwargs = synth_kwargs or {}
    KEEP = ["wav_audio_path","is_disfluent","disfluency_type",
            "speaker","source","intensidad","severidad",
            "num_disfluencias","chunk_start","chunk_end","chunk_text"]

    if source == "real":
        df = load_real_data()
    elif source == "synthetic":
        df = discover_synthetic_clips(**synth_kwargs)
    elif source == "both":
        real  = load_real_data()
        synth = discover_synthetic_clips(**synth_kwargs)
        df    = pd.concat([real, synth], ignore_index=True)
    else:
        raise ValueError(f"source debe ser 'real', 'synthetic' o 'both'. Recibido: {source}")

    # Asegurar columnas opcionales
    for c in KEEP:
        if c not in df.columns:
            df[c] = None

    df = df[KEEP].dropna(subset=["wav_audio_path"]).reset_index(drop=True)
    print(f"\n[load_data] Fuente='{source}' → {len(df)} clips totales "
          f"({df['is_disfluent'].sum()} disfluentes)")
    return df

# Preprocesamiento de audio

Carga y normaliza un clip de audio.


*   Resamplea a TARGET_SR
*   Convierte a mono
*   Normaliza amplitud a [-1, 1]

Devuelve None si el archivo no existe o hay error de lectura.

In [ ]:
def load_audio(path: str, sr: int = TARGET_SR) -> np.ndarray | None:
    if not Path(path).exists():
        return None
    try:
        y, orig_sr = librosa.load(path, sr=None, mono=True, res_type="kaiser_fast")
        if orig_sr != sr:
            y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)
        max_amp = np.abs(y).max()
        if max_amp > 0:
            y = y / max_amp
        return y.astype(np.float32)
    except Exception as e:
        print(f"    [Audio] Error {path}: {e}")
        return None


# Extraccion de caracteristicas principales

## Extraer vector MFCC de un clip

In [ ]:
_w2v_proc  = None
_w2v_model = None

In [ ]:
def extract_mfcc(audio_path: str, n_mfcc: int = N_MFCC) -> np.ndarray | None:
    y = load_audio(audio_path)
    if y is None or len(y) < 512:
        return None
    mfccs = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=n_mfcc)
    return np.hstack([mfccs.mean(1), mfccs.std(1)])

## Convertir a embedding usando wav2vec

In [ ]:
def _get_wav2vec():
    """Singleton: carga Wav2Vec2 una sola vez."""
    global _w2v_proc, _w2v_model
    if _w2v_model is None:
        print(f"    [Wav2Vec2] Cargando {WAV2VEC_ID} → {DEVICE}")
        _w2v_proc  = Wav2Vec2Processor.from_pretrained(WAV2VEC_ID)
        _w2v_model = Wav2Vec2Model.from_pretrained(WAV2VEC_ID).to(DEVICE)
        _w2v_model.eval()
    return _w2v_proc, _w2v_model

In [ ]:
def extract_wav2vec(audio_path: str) -> np.ndarray | None:
    """
    Extrae embedding Wav2Vec2 de un clip.

    Proceso:
      1. Cargar y preparar audio (load_audio)
      2. Tokenizar con Wav2Vec2Processor
      3. Forward pass sin gradiente → last_hidden_state [1, T, 768]
      4. Promedio sobre eje temporal → vector (768,)
    """
    y = load_audio(audio_path)
    if y is None or len(y) < 512:
        return None
    proc, mdl = _get_wav2vec()
    try:
        inputs = proc(y, sampling_rate=TARGET_SR, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = mdl(**inputs)
        return out.last_hidden_state.mean(1).squeeze().cpu().numpy()
    except Exception as e:
        print(f"    [Wav2Vec] Error {audio_path}: {e}")
        return None


## Construccion de matrices

In [ ]:
META_COLS = ["disfluency_type", "intensidad", "severidad",
             "speaker", "source", "num_disfluencias",
             "chunk_start", "chunk_end", "chunk_text"]

In [ ]:
def build_dataset(
    df: pd.DataFrame,
    feature: str = "mfcc",
    cache_tag: str = "",
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Construye matrices X, y y metadata para entrenamiento.

    Args:
        df         : DataFrame con columna wav_audio_path e is_disfluent
        feature    : "mfcc" o "wav2vec"
        cache_tag  : Si no vacío, guarda/lee cache en CACHE_DIR/<cache_tag>_<feature>.npz

    Returns:
        X    : (n_samples, n_features)
        y    : (n_samples,) int binario
        meta : DataFrame (n_samples, len(META_COLS))
    """
    cache_path = CACHE_DIR / f"{cache_tag}_{feature}.npz" if cache_tag else None

    if cache_path and cache_path.exists():
        print(f"    [Cache] Cargando {cache_path.name}")
        data = np.load(cache_path, allow_pickle=True)
        X    = data["X"]
        y    = data["y"]
        meta = pd.DataFrame(data["meta"].tolist())
        print(f"    X={X.shape}  y={y.shape}")
        return X, y, meta

    fn = extract_mfcc if feature == "mfcc" else extract_wav2vec
    desc = f"Extrayendo {feature.upper()}"

    Xlist, ylist, mlist = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        feats = fn(row["wav_audio_path"])
        if feats is not None:
            Xlist.append(feats)
            ylist.append(int(row["is_disfluent"]))
            mlist.append({c: row.get(c) for c in META_COLS})

    X    = np.array(Xlist, dtype=np.float32)
    y    = np.array(ylist, dtype=np.int32)
    meta = pd.DataFrame(mlist)

    if cache_path:
        np.savez(cache_path, X=X, y=y, meta=np.array(mlist))
        print(f"    [Cache] Guardado en {cache_path.name}")

    print(f"    X={X.shape}  y={y.shape}  ({y.sum()} disfluentes)")
    return X, y, meta

# Dividir en sets de entrenamiento , validacion y prueba

In [ ]:

def make_splits(
    X: np.ndarray,
    y: np.ndarray,
    meta: pd.DataFrame = None,
    seed: int = RANDOM_STATE,
) -> dict:
    """
    Split estratificado: 34% train / 33% val / 33% test.
    (Igual que en el notebook original: test_size=0.66, luego 0.5)
    """
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=0.66, stratify=y, random_state=seed)
    va, te  = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)

    s = dict(
        X_train=X[tr], y_train=y[tr],
        X_val  =X[va], y_val  =y[va],
        X_test =X[te], y_test =y[te],
    )
    if meta is not None:
        s["meta_train"] = meta.iloc[tr].reset_index(drop=True)
        s["meta_val"]   = meta.iloc[va].reset_index(drop=True)
        s["meta_test"]  = meta.iloc[te].reset_index(drop=True)

    print("\n    [Splits]")
    for k in ("train","val","test"):
        yy = s[f"y_{k}"]
        print(f"      {k:5s}: {len(yy):5d}  |  dis={yy.sum():4d} ({yy.mean():.1%})")
    return s


In [ ]:

def make_speaker_holdout_splits(
    X: np.ndarray,
    y: np.ndarray,
    meta: pd.DataFrame,
    holdout_frac: float = 0.25,
    seed: int = RANDOM_STATE,
) -> dict:
    """
    Split por speaker: los hablantes de test NO aparecen en train/val.
    Usado en Exp 6 y 7 para medir generalización cross-speaker.
    """
    speakers    = meta["speaker"].unique()
    n_hold      = max(1, int(len(speakers) * holdout_frac))
    rng         = np.random.default_rng(seed)
    held        = set(rng.choice(speakers, n_hold, replace=False))
    seen        = set(speakers) - held

    m_tr = meta["speaker"].isin(seen).values
    m_te = meta["speaker"].isin(held).values

    X_seen, y_seen = X[m_tr], y[m_tr]
    idx = np.arange(len(y_seen))
    tr, va = train_test_split(idx, test_size=0.25, stratify=y_seen, random_state=seed)

    s = dict(
        X_train=X_seen[tr],  y_train=y_seen[tr],
        X_val  =X_seen[va],  y_val  =y_seen[va],
        X_test =X[m_te],     y_test =y[m_te],
        meta_train=meta[m_tr].iloc[tr].reset_index(drop=True),
        meta_val  =meta[m_tr].iloc[va].reset_index(drop=True),
        meta_test =meta[m_te].reset_index(drop=True),
        held_speakers=list(held),
        seen_speakers=list(seen),
    )
    print(f"\n    [Speaker-Holdout] Vistos={len(seen)}  Holdout={len(held)}")
    for k in ("train","val","test"):
        yy = s[f"y_{k}"]
        print(f"      {k:5s}: {len(yy):5d}  |  dis={yy.sum():4d} ({yy.mean():.1%})")
    return s


# Entrenamiento

In [ ]:
def train_lr(X_tr: np.ndarray, y_tr: np.ndarray) -> LogisticRegression:
    """
    Regresión Logística con StandardScaler previo.
    El scaler se adjunta al modelo como atributo _scaler
    para ser aplicado automáticamente en inferencia.
    """
    sc = StandardScaler()
    Xs = sc.fit_transform(X_tr)
    m  = LogisticRegression(max_iter=1000, solver="liblinear", random_state=RANDOM_STATE)
    m.fit(Xs, y_tr)
    m._scaler = sc
    print("    ✓ Logistic Regression entrenada")
    return m


In [ ]:
def train_rf(X_tr: np.ndarray, y_tr: np.ndarray) -> RandomForestClassifier:
    """Random Forest con 100 árboles, todos los núcleos disponibles."""
    m = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
    m.fit(X_tr, y_tr)
    print("    ✓ Random Forest entrenado")
    return m


In [ ]:
def _infer(model, X: np.ndarray):
    """Aplica scaler si existe, predice clase y probabilidades."""
    if hasattr(model, "_scaler"):
        X = model._scaler.transform(X)
    yp  = model.predict(X)
    ypr = model.predict_proba(X)[:,1] if hasattr(model,"predict_proba") else yp.astype(float)
    return yp, ypr

In [ ]:
def save_model(model, name: str):
    p = MODELS_DIR / f"{name}.joblib"
    joblib.dump(model, p)
    print(f"    [Save] {p.name}")

# Evaluacion y metricas

    Métricas obligatorias:
      ├─ Accuracy
      ├─ Macro F1          ← métrica principal para desbalance
      ├─ F1 por clase      (fluido / disfluente)
      ├─ Precision y Recall
      ├─ ROC-AUC
      └─ Matriz de Confusión  (guardada como figura)

    Análisis opcionales (si meta disponible):
      ├─ F1 por tipo de disfluencia  → ¿qué tipo mejora más?
      ├─ F1 por severidad            → ¿qué severidad funciona mejor?
      └─ F1 por intensidad           → ¿qué intensidad funciona mejor?

In [ ]:
def evaluate(
    model,
    X: np.ndarray,
    y_true: np.ndarray,
    meta: pd.DataFrame = None,
    model_name: str = "Model",
    split_name: str = "test",
    save_figs:  bool = True,
) -> dict:

    y_pred, y_prob = _infer(model, X)

    acc      = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_per   = f1_score(y_true, y_pred, average=None,   zero_division=0)
    prec     = precision_score(y_true, y_pred, zero_division=0)
    rec      = recall_score(y_true, y_pred, zero_division=0)
    auc      = (roc_auc_score(y_true, y_prob)
                if len(np.unique(y_true)) > 1 else float("nan"))

    tag = f"{model_name} | {split_name.upper()}"
    sep = "═" * 60
    print(f"\n{sep}")
    print(f"  {tag}")
    print("─" * 60)
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Macro F1          : {macro_f1:.4f}  ← PRINCIPAL")
    print(f"  F1 clase Fluido   : {f1_per[0]:.4f}")
    print(f"  F1 clase Disfluente: {f1_per[1]:.4f}")
    print(f"  Precision         : {prec:.4f}")
    print(f"  Recall            : {rec:.4f}")
    print(f"  ROC-AUC           : {auc:.4f}")
    print("─" * 60)
    print(classification_report(y_true, y_pred,
          target_names=["Fluido","Disfluente"], zero_division=0))

    result = dict(
        model=model_name, split=split_name,
        accuracy=acc, macro_f1=macro_f1,
        f1_fluido=float(f1_per[0]),
        f1_disfluente=float(f1_per[1]),
        precision=prec, recall=rec, roc_auc=auc,
    )

    safe = model_name.replace(" ","_")

    if save_figs:
        _plot_cm(y_true, y_pred, model_name, split_name, safe)

    # Análisis por subgrupos (si hay metadata)
    if meta is not None:
        for col in ("disfluency_type","severidad","intensidad"):
            if col in meta.columns and meta[col].notna().any():
                print(f"\n  ── Análisis por {col} ──")
                result[f"by_{col}"] = _f1_by_group(
                    y_true, y_pred, meta, col, safe, split_name, save_figs
                )

    return result

In [ ]:
def _plot_cm(y_true, y_pred, model_name, split_name, safe_name):
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Fluido","Disfluente"])
    fig, ax = plt.subplots(figsize=(5,4))
    disp.plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"Conf Matrix: {model_name}\n{split_name.upper()}")
    plt.tight_layout()
    p = FIGS_DIR / f"cm_{safe_name}_{split_name}.png"
    plt.savefig(p, dpi=120); plt.close()
    print(f"  [Fig] {p.name}")

In [ ]:
def _f1_by_group(y_true, y_pred, meta, col, safe_name, split, save_figs) -> dict:
    """F1 macro por cada valor del grupo + gráfico de barras."""
    groups = sorted(meta[col].dropna().unique())
    out = {}
    for g in groups:
        mask = (meta[col] == g).values
        if mask.sum() == 0:
            continue
        f1 = f1_score(y_true[mask], y_pred[mask], average="macro", zero_division=0)
        out[g] = round(f1, 4)
        print(f"    {col}={g:12s}  Macro F1={f1:.4f}  n={mask.sum()}")

    if save_figs and out:
        fig, ax = plt.subplots(figsize=(8,4))
        ks, vs = zip(*out.items())
        bars = ax.bar(ks, vs, color=sns.color_palette("Set2", len(ks)))
        ax.set_ylim(0,1); ax.set_title(f"F1 por {col}"); ax.set_ylabel("Macro F1")
        for bar, v in zip(bars, vs):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}",
                    ha="center", fontsize=9)
        plt.tight_layout()
        p = FIGS_DIR / f"by_{col}_{safe_name}_{split}.png"
        plt.savefig(p, dpi=120); plt.close()
        print(f"  [Fig] {p.name}")
    return out


In [ ]:
def evaluate_splits(
    model,
    splits: dict,
    model_name: str,
    save_figs: bool = True,
) -> dict:
    """Evalúa en val y test. Devuelve {'val': {...}, 'test': {...}}."""
    return {
        sp: evaluate(
            model,
            splits[f"X_{sp}"], splits[f"y_{sp}"],
            meta      = splits.get(f"meta_{sp}"),
            model_name= model_name,
            split_name= sp,
            save_figs = save_figs,
        )
        for sp in ("val","test")
    }


# Prediccion e Inferencia

 Clasifica un único archivo de audio.

    Args:
        audio_path : Ruta al .wav
        model      : Modelo entrenado (LR o RF, con _scaler si aplica)
        feature    : "mfcc" o "wav2vec"

    Returns:
        dict con:
          label      : 0 (fluido) | 1 (disfluente)
          label_text : "Fluido" | "Disfluente"
          probability: probabilidad de clase 1
          feature    : feature usada

In [ ]:
def predict_audio(
    audio_path: str,
    model,
    feature: str = "mfcc",
) -> dict:

    fn    = extract_mfcc if feature == "mfcc" else extract_wav2vec
    feats = fn(audio_path)
    if feats is None:
        return {"label": None, "label_text": "ERROR", "probability": None}

    y_pred, y_prob = _infer(model, feats.reshape(1,-1))
    return {
        "label":       int(y_pred[0]),
        "label_text":  "Disfluente" if y_pred[0] == 1 else "Fluido",
        "probability": float(y_prob[0]),
        "feature":     feature,
    }

In [ ]:
def predict_batch(
    audio_paths: list[str],
    model,
    feature: str = "mfcc",
) -> pd.DataFrame:
    records = []
    for p in tqdm(audio_paths, desc="Prediciendo"):
        r = predict_audio(p, model, feature)
        r["wav_audio_path"] = p
        records.append(r)
    return pd.DataFrame(records)


# Fine tunning  - Experimento 7

In [ ]:
class _AudioDS(torch.utils.data.Dataset):
    """Dataset PyTorch para fine-tuning: carga audio y etiqueta."""
    def __init__(self, paths, labels, processor, max_sec=5):
        self.paths, self.labels = paths, labels
        self.proc = processor
        self.maxl = TARGET_SR * max_sec

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        audio_data = load_audio(self.paths[i])
        if audio_data is None:
            y = np.zeros(TARGET_SR, dtype=np.float32)
        else:
            y = audio_data
        if len(y) > self.maxl:
            y = y[:self.maxl]
        enc = self.proc(y, sampling_rate=TARGET_SR, return_tensors="pt",
                        padding=True, max_length=self.maxl, truncation=True)
        return {"input_values": enc.input_values.squeeze(0),
                "label": torch.tensor(self.labels[i], dtype=torch.long)}


In [ ]:
def _collate(batch):
    ml = max(b["input_values"].shape[0] for b in batch)
    iv = torch.zeros(len(batch), ml)
    am = torch.zeros(len(batch), ml)
    lb = []
    for i, b in enumerate(batch):
        l = b["input_values"].shape[0]
        iv[i,:l] = b["input_values"]; am[i,:l] = 1
        lb.append(b["label"])
    return {"input_values":iv, "attention_mask":am, "labels":torch.stack(lb)}

In [ ]:
def _freeze(model, n):
    if n == 0: return
    for p in model.wav2vec2.feature_extractor.parameters(): p.requires_grad = False
    for i, layer in enumerate(model.wav2vec2.encoder.layers):
        if i < n:
            for p in layer.parameters(): p.requires_grad = False

In [ ]:
def _run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot_loss, preds, trues = 0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for b in loader:
            iv = b["input_values"].to(DEVICE)
            am = b["attention_mask"].to(DEVICE)
            lb = b["labels"].to(DEVICE)
            out = model(input_values=iv, attention_mask=am, labels=lb)
            if train:
                optimizer.zero_grad(); out.loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            tot_loss += out.loss.item()
            preds.extend(out.logits.argmax(1).cpu().tolist())
            trues.extend(lb.cpu().tolist())
    f1 = f1_score(trues, preds, average="macro", zero_division=0)
    return tot_loss / len(loader), f1, preds, trues

    Experimento 7 — Fine-tuning Wav2Vec2 en dos fases:

      Fase 1 (sintético): el encoder aprende patrones acústicos de
        disfluencia desde datos abundantes pero artificiales.
        → LR moderada, encoder descongelado (p1_freeze=0).

      Fase 2 (real): partiendo del checkpoint de Fase 1, se ajustan
        solo las capas superiores con datos reales escasos.
        → LR baja (1e-5), primeras p2_freeze capas congeladas,
          early stopping por Macro F1 en validación real.

    El modelo resultante combina la riqueza acústica del preentrenamiento
    sintético con la autenticidad del fine-tuning real.

In [ ]:
def finetune_wav2vec2(
    synth_df: pd.DataFrame,
    real_df:  pd.DataFrame,
    save_path: str = str(MODELS_DIR / "wav2vec2_ft"),
    p1_epochs: int = 5,   p1_lr: float = 1e-4,
    p1_batch:  int = 8,   p1_freeze: int = 0,
    p2_epochs: int = 10,  p2_lr: float = 1e-5,
    p2_batch:  int = 8,   p2_freeze: int = 8,
    patience:  int = 3,
) -> dict:

    proc = Wav2Vec2Processor.from_pretrained(WAV2VEC_ID)

    # ─ Splits sintéticos ─
    y_s   = synth_df["is_disfluent"].values
    idx   = np.arange(len(synth_df))
    tr,va = train_test_split(idx, test_size=0.2, stratify=y_s, random_state=RANDOM_STATE)
    dl_s_tr = torch.utils.data.DataLoader(
        _AudioDS(synth_df["wav_audio_path"].tolist(), y_s.tolist(), proc),
        batch_size=p1_batch, shuffle=True, collate_fn=_collate)
    dl_s_va = torch.utils.data.DataLoader(
        _AudioDS(synth_df.iloc[va]["wav_audio_path"].tolist(), y_s[va].tolist(), proc),
        batch_size=p1_batch, shuffle=False, collate_fn=_collate)

    # ─ Splits reales ─
    y_r   = real_df["is_disfluent"].values
    idx_r = np.arange(len(real_df))
    tr_r,tmp = train_test_split(idx_r, test_size=0.5, stratify=y_r, random_state=RANDOM_STATE)
    va_r,te_r= train_test_split(tmp,   test_size=0.5, stratify=y_r[tmp], random_state=RANDOM_STATE)
    mk_dl = lambda ids, shuf, bs: torch.utils.data.DataLoader(
        _AudioDS(real_df.iloc[ids]["wav_audio_path"].tolist(), y_r[ids].tolist(), proc),
        batch_size=bs, shuffle=shuf, collate_fn=_collate)
    dl_r_tr = mk_dl(tr_r, True,  p2_batch)
    dl_r_va = mk_dl(va_r, False, p2_batch)
    dl_r_te = mk_dl(te_r, False, p2_batch)

    print(f"\n  Sintético → train:{len(tr)} val:{len(va)}")
    print(f"  Real      → train:{len(tr_r)} val:{len(va_r)} test:{len(te_r)}")

    # ════ FASE 1: Pre-entrenamiento sintético ════
    print(f"\n  [F1] Pre-entrenamiento sintético ({p1_epochs} épocas) …")
    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        WAV2VEC_ID, num_labels=2).to(DEVICE)
    _freeze(model, p1_freeze)
    opt1  = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=p1_lr, weight_decay=0.01)

    h1 = {"loss":[], "val_f1":[]}
    for ep in range(p1_epochs):
        loss, _, _, _ = _run_epoch(model, dl_s_tr, opt1)
        _, vf1, _, _  = _run_epoch(model, dl_s_va)
        h1["loss"].append(loss); h1["val_f1"].append(vf1)
        print(f"    Ep {ep+1}/{p1_epochs}  loss={loss:.4f}  val_f1={vf1:.4f}")

    # ════ FASE 2: Fine-tuning real ════
    print(f"\n  [F2] Fine-tuning real ({p2_epochs} épocas, patience={patience}) …")
    _freeze(model, p2_freeze)
    opt2 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=p2_lr, weight_decay=0.01)

    best_f1, best_sd, pat_ct = 0.0, None, 0
    h2 = {"loss":[], "val_f1":[]}
    for ep in range(p2_epochs):
        loss, _, _, _ = _run_epoch(model, dl_r_tr, opt2)
        _, vf1, _, _  = _run_epoch(model, dl_r_va)
        h2["loss"].append(loss); h2["val_f1"].append(vf1)
        flag = ""
        if vf1 > best_f1:
            best_f1 = vf1; best_sd = copy.deepcopy(model.state_dict())
            pat_ct = 0; flag = " ← mejor"
        else:
            pat_ct += 1
        print(f"    Ep {ep+1}/{p2_epochs}  loss={loss:.4f}  val_f1={vf1:.4f}{flag}")
        if pat_ct >= patience:
            print(f"    Early stopping en época {ep+1}"); break

    model.load_state_dict(best_sd)
    model.save_pretrained(save_path)
    proc.save_pretrained(save_path)
    print(f"  Modelo FT guardado: {save_path}")

    # Evaluación en test real
    _, f1_te, preds, trues = _run_epoch(model, dl_r_te)
    print(f"\n  [Eval FT] Test Macro F1 = {f1_te:.4f}")
    print(classification_report(trues, preds,
          target_names=["Fluido","Disfluente"], zero_division=0))

    _plot_ft_curves(h1, h2)
    return {"test_macro_f1": f1_te, "h1": h1, "h2": h2}

In [ ]:
def _plot_ft_curves(h1, h2):
    fig, axes = plt.subplots(1,2,figsize=(13,5))
    for ax, h, title in zip(axes, [h1,h2],
        ["Fase 1 — Pre-entrenamiento sintético",
         "Fase 2 — Fine-tuning real"]):
        ep = range(1, len(h["loss"])+1)
        ax.plot(ep, h["loss"],    "b-o", label="Train Loss")
        ax2 = ax.twinx()
        ax2.plot(ep, h["val_f1"],"r-s", label="Val Macro F1")
        ax.set(xlabel="Época", ylabel="Loss", title=title)
        ax2.set_ylabel("Macro F1", color="r")
        lines = ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
        labs  = ax.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
        ax.legend(lines, labs)
    plt.tight_layout()
    p = FIGS_DIR / "exp7_ft_curves.png"
    plt.savefig(p, dpi=120); plt.close()
    print(f"  [Fig] {p.name}")

## Comparacion Global

In [ ]:
def plot_experiment_comparison(all_results: dict, metric: str = "macro_f1"):
    """
    Gráfico de barras agrupadas: todos los experimentos × todos los modelos.
    all_results: {exp_name: {model_name: {split: {metric: value}}}}
    """
    rows = []
    for exp, models in all_results.items():
        for mdl, splits in models.items():
            v = splits.get("test", {}).get(metric, float("nan"))
            rows.append({"Experimento": exp, "Modelo": mdl, metric: v})
    df = pd.DataFrame(rows).dropna(subset=[metric])
    if df.empty:
        return df

    fig, ax = plt.subplots(figsize=(14,6))
    sns.barplot(data=df, x="Experimento", y=metric, hue="Modelo",
                palette="tab10", ax=ax)
    ax.set_title(f"Comparación global — {metric}")
    ax.set_ylim(0,1); ax.set_ylabel(metric)
    ax.legend(loc="lower right")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    p = FIGS_DIR / f"global_comparison_{metric}.png"
    plt.savefig(p, dpi=120); plt.close()
    print(f"[Fig] Comparación global → {p.name}")
    return df

# Experimentos

In [ ]:
def _run_baseline(df, tag, features=("mfcc","wav2vec"), use_holdout=False):
    """
    Entrena LR + RF para cada feature especificada.
    Devuelve dict: {model_name: {val: {...}, test: {...}}}
    """
    results = {}
    meta_all = None

    for feat in features:
        print(f"\n  ── {feat.upper()} ──")
        X, y, meta = build_dataset(df, feature=feat, cache_tag=tag)

        if use_holdout and "speaker" in meta.columns:
            splits = make_speaker_holdout_splits(X, y, meta)
        else:
            splits = make_splits(X, y, meta)

        for name, trainer in [("LR", train_lr), ("RF", train_rf)]:
            mname = f"{name}-{feat.upper()}-{tag}"
            print(f"\n  Entrenando {mname} …")
            m = trainer(splits["X_train"], splits["y_train"])
            results[mname] = evaluate_splits(m, splits, mname)
            save_model(m, mname)

    return results

Experimento 1

In [ ]:
def experiment_1(features=("mfcc","wav2vec")) -> dict:
    """Baseline: solo datos reales."""
    print("\n" + "╔"+"═"*58+"╗")
    print("║  EXPERIMENTO 1 — Baseline real only" + " "*22 + "║")
    print("╚"+"═"*58+"╝")
    df = load_data(source="real")
    return _run_baseline(df, "exp1_real", features)

Experimento 2

In [ ]:
def experiment_2(features=("mfcc","wav2vec")) -> dict:
    """Real + sintético sin filtros."""
    print("\n" + "╔"+"═"*58+"╗")
    print("║  EXPERIMENTO 2 — Real + sintético simple" + " "*18 + "║")
    print("╚"+"═"*58+"╝")
    df = load_data(source="both")
    return _run_baseline(df, "exp2_both", features)

Experimento 3

In [ ]:
def experiment_3(
    tipos=None, severidades=None, intensidades=None,
    features=("mfcc","wav2vec"),
) -> dict:
    """Real + sintético con control total de tipo/severidad/intensidad."""
    print("\n" + "╔"+"═"*58+"╗")
    print("║  EXPERIMENTO 3 — Real + sintético controlado" + " "*13 + "║")
    print("╚"+"═"*58+"╝")
    df = load_data(source="both", synth_kwargs=dict(
        tipo_filter=tipos,
        severidad_filter=severidades,
        intensidad_filter=intensidades,
    ))
    tag = "exp3_ctrl"
    return _run_baseline(df, tag, features)

Experimento 4

In [ ]:
def experiment_4(intensidad_fija="media", features=("mfcc",)) -> dict:
    """
    Ablación por severidad — fija intensidad, varía severidad.
    Responde: ¿qué nivel de severidad de disfluencia ayuda más al modelo?
    """
    print("\n" + "╔"+"═"*58+"╗")
    print("║  EXPERIMENTO 4 — Ablación por severidad" + " "*19 + "║")
    print("╚"+"═"*58+"╝")
    results = {}
    for sev in SYNTH_SEV:
        print(f"\n  → Severidad: {sev}  (intensidad fija={intensidad_fija})")
        df = load_data(source="both", synth_kwargs=dict(
            intensidad_filter=[intensidad_fija],
            severidad_filter=[sev],
        ))
        r = _run_baseline(df, f"exp4_{sev}", features)
        results[sev] = r
    return results

Experimento 5

In [ ]:
def experiment_5(severidad_fija="moderado", features=("mfcc",)) -> dict:
    """
    Ablación por intensidad — fija severidad, varía intensidad.
    Responde: ¿qué nivel de intensidad (densidad de disfluencias) funciona mejor?
    """
    print("\n" + "╔"+"═"*58+"╗")
    print("║  EXPERIMENTO 5 — Ablación por intensidad" + " "*18 + "║")
    print("╚"+"═"*58+"╝")
    results = {}
    for inten in SYNTH_INTEN:
        print(f"\n  → Intensidad: {inten}  (severidad fija={severidad_fija})")
        df = load_data(source="both", synth_kwargs=dict(
            intensidad_filter=[inten],
            severidad_filter=[severidad_fija],
        ))
        r = _run_baseline(df, f"exp5_{inten}", features)
        results[inten] = r
    return results

Experimento 6

In [ ]:
def experiment_6(
    n_samples_fixed: int = 200,
    speaker_fractions: list = None,
    features: tuple = ("mfcc",),
) -> dict:
    """
    Diversidad de hablantes vs cantidad de datos.
    Fija total de muestras sintéticas, varía número de speakers.
    Responde: ¿importa más la diversidad de voces o la cantidad?
    """
    print("\n" + "╔"+"═"*58+"╗")
    print("║  EXPERIMENTO 6 — Diversidad de hablantes" + " "*18 + "║")
    print("╚"+"═"*58+"╝")

    all_synth = discover_synthetic_clips()
    all_sp    = sorted(all_synth["speaker"].unique())
    n_total   = len(all_sp)

    if speaker_fractions is None:
        speaker_fractions = [0.1, 0.25, 0.5, 0.75, 1.0]

    results = {}
    diversity_log = []

    real_df = load_real_data()

    for frac in speaker_fractions:
        n_sp   = max(1, int(n_total * frac))
        chosen = all_sp[:n_sp]
        synth_sub = all_synth[all_synth["speaker"].isin(chosen)]

        # Subsamplear para mantener n_samples_fixed total
        synth_sub = synth_sub.sample(
            min(n_samples_fixed, len(synth_sub)),
            replace=len(synth_sub) < n_samples_fixed,
            random_state=RANDOM_STATE,
        )
        df = pd.concat([real_df, synth_sub], ignore_index=True)
        print(f"\n  → {n_sp} speakers  |  {len(synth_sub)} muestras sintéticas")

        tag = f"exp6_sp{n_sp}"
        r   = _run_baseline(df, tag, features, use_holdout=True)
        results[f"sp_{n_sp}"] = r

        # Extraer macro_f1 de test del primer modelo
        first_key = next(iter(r))
        f1_te = r[first_key].get("test",{}).get("macro_f1", float("nan"))
        diversity_log.append({"n_speakers": n_sp, "macro_f1": f1_te})

    # Curva diversidad
    _plot_diversity_curve(diversity_log)
    return results

In [ ]:
def _plot_diversity_curve(log: list):
    df = pd.DataFrame(log)
    fig, ax = plt.subplots(figsize=(8,5))
    ax.plot(df["n_speakers"], df["macro_f1"], "o-", color="steelblue", lw=2)
    ax.set(xlabel="Nº de hablantes sintéticos en train",
           ylabel="Macro F1 (test)", title="Exp 6 — Diversidad de hablantes")
    plt.tight_layout()
    p = FIGS_DIR / "exp6_diversity_curve.png"
    plt.savefig(p, dpi=120); plt.close()
    print(f"  [Fig] {p.name}")

In [ ]:
''' def experiment_7(
    run_wav2vec_ft: bool = True,
    features: tuple = ("mfcc",),
) -> dict:
    """
    Sintético → Fine-tuning con reales.

    Estrategias:
      • LR/RF: re-entrenamiento ponderado (peso=0.3 sintético, 1.0 real)
      • Wav2Vec2: fine-tuning neuronal en dos fases (ver finetune_wav2vec2)
    """
    print("\n" + "╔"+"═"*58+"╗")
    print("║  EXPERIMENTO 7 — Sintético → Fine-Tuning con reales" + " "*7 + "║")
    print("╚"+"═"*58+"╝")

    real_df  = load_real_data()
    synth_df = discover_synthetic_clips()
    results  = {}

    for feat in features:
        print(f"\n  ── {feat.upper()} ──")
        X_sy, y_sy, _       = build_dataset(synth_df, feature=feat, cache_tag="exp7_synth")
        X_re, y_re, meta_re = build_dataset(real_df,  feature=feat, cache_tag="exp7_real")
        splits_re = make_splits(X_re, y_re, meta_re)

        # Ponderación: real tiene peso 1.0, sintético 0.3
        w_sy = np.full(len(y_sy), 0.3)
        w_re = np.full(len(y_re), 1.0)
        X_c  = np.vstack([X_sy, X_re])
        y_c  = np.concatenate([y_sy, y_re])
        w_c  = np.concatenate([w_sy, w_re])

        # LR ponderada
        sc   = StandardScaler(); Xs = sc.fit_transform(X_c)
        lr_ft = LogisticRegression(max_iter=1000, solver="liblinear",
                                   random_state=RANDOM_STATE)
        lr_ft.fit(Xs, y_c, sample_weight=w_c)
        lr_ft._scaler = s
        name = f"LR-{feat.upper()}-FT"
        results[name] = evaluate_splits(lr_ft, splits_re, name)
        save_model(lr_ft, name)

        # RF incremental (sintético + real)
        rf_sy = train_rf(X_sy, y_sy)
        rf_re = train_rf(X_re, y_re)
        rf_ft = copy.deepcopy(rf_sy)
        rf_ft.estimators_ += rf_re.estimators_
        rf_ft.n_estimators = len(rf_ft.estimators_)
        name = f"RF-{feat.upper()}-FT"
        results[name] = evaluate_splits(rf_ft, splits_re, name)
        save_model(rf_ft, name)

    # Wav2Vec2 fine-tuning neuronal
    if run_wav2vec_ft:
        print("\n  ── WAV2VEC2 Fine-Tuning neuronal ──")
        ft_result = finetune_wav2vec2(synth_df, real_df)
        results["Wav2Vec2-FT"] = {"test": ft_result}

    return results '''

' def experiment_7(\n    run_wav2vec_ft: bool = True,\n    features: tuple = ("mfcc",),\n) -> dict:\n    """\n    Sintético → Fine-tuning con reales.\n\n    Estrategias:\n      • LR/RF: re-entrenamiento ponderado (peso=0.3 sintético, 1.0 real)\n      • Wav2Vec2: fine-tuning neuronal en dos fases (ver finetune_wav2vec2)\n    """\n    print("\n" + "╔"+"═"*58+"╗")\n    print("║  EXPERIMENTO 7 — Sintético → Fine-Tuning con reales" + " "*7 + "║")\n    print("╚"+"═"*58+"╝")\n\n    real_df  = load_real_data()\n    synth_df = discover_synthetic_clips()\n    results  = {}\n\n    for feat in features:\n        print(f"\n  ── {feat.upper()} ──")\n        X_sy, y_sy, _       = build_dataset(synth_df, feature=feat, cache_tag="exp7_synth")\n        X_re, y_re, meta_re = build_dataset(real_df,  feature=feat, cache_tag="exp7_real")\n        splits_re = make_splits(X_re, y_re, meta_re)\n\n        # Ponderación: real tiene peso 1.0, sintético 0.3\n        w_sy = np.full(len(y_sy), 0.3)\n        w_re = n

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from torch.cuda.amp import GradScaler, autocast

# Importar del pipeline principal
# The following functions and variables are already defined in the notebook context and do not need to be imported.
# from disfluency_pipeline_v3 import (
#     load_real_data, discover_synthetic_clips,
#     build_dataset, make_splits,
#     evaluate_splits, save_model,
#     load_audio, MODELS_DIR, FIGS_DIR, CACHE_DIR,
#     TARGET_SR, WAV2VEC_ID, RANDOM_STATE, DEVICE,
# )

import soundfile as sf
import librosa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm


# ══════════════════════════════════════════════════════════════════
# UTILIDADES INTERNAS
# ══════════════════════════════════════════════════════════════════

def _subsample_synthetic_fluent(
    synth_df,
    real_df,
    target_ratio: float = 2.0,
    seed: int = RANDOM_STATE,
):
    """
    Submuestrea los clips sintéticos FLUIDOS para que la proporción
    fluido/disfluente del corpus combinado no supere target_ratio.

    Problema original: 11571 sintéticos fluidos + 1382 disfluentes reales
    → clase fluida representa 88% del corpus combinado.

    Con target_ratio=2.0 se mantiene máximo 2 fluidos por cada disfluente.

    Args:
        synth_df     : DataFrame sintético completo
        real_df      : DataFrame real completo
        target_ratio : máximo ratio fluido/disfluente permitido

    Returns:
        synth_df_balanced : sintético submuestreado
    """
    n_dis_real  = int(real_df["is_disfluent"].sum())
    n_dis_synth = int(synth_df["is_disfluent"].sum())
    n_dis_total = n_dis_real + n_dis_synth

    n_flu_max   = int(n_dis_total * target_ratio)

    synth_flu   = synth_df[synth_df["is_disfluent"] == 0]
    synth_dis   = synth_df[synth_df["is_disfluent"] == 1]

    # Fluidos reales
    n_flu_real  = int((real_df["is_disfluent"] == 0).sum())
    n_flu_synth_max = max(0, n_flu_max - n_flu_real)

    if len(synth_flu) > n_flu_synth_max:
        synth_flu = synth_flu.sample(n_flu_synth_max, random_state=seed)
        print(f"    [Balance] Sintético fluido: {len(synth_flu)+len(synth_dis[synth_dis['is_disfluent']==0])} → {n_flu_synth_max}")

    import pandas as pd
    return pd.concat([synth_flu, synth_dis], ignore_index=True)


# ══════════════════════════════════════════════════════════════════
# MODELOS BASELINE CORREGIDOS
# ══════════════════════════════════════════════════════════════════

def _train_lr_ft_v2(X_sy, y_sy, X_re, y_re, splits_re):
    """
    LR con ponderación adaptativa basada en frecuencia de clase.

    Correcciones:
      - Peso por muestra calculado con compute_sample_weight('balanced')
        sobre el dataset combinado → compensa automáticamente cualquier
        desbalance sin necesidad de fijar 0.3/1.0 a mano.
      - Adicionalmente, las muestras reales reciben un multiplicador
        extra (x3) para priorizar el dominio real sobre el sintético.
    """
    # Pesos base balanceados por clase
    X_c = np.vstack([X_sy, X_re])
    y_c = np.concatenate([y_sy, y_re])
    w_c = compute_sample_weight("balanced", y_c)

    # Multiplicador de dominio: real x3, sintético x1
    domain = np.concatenate([
        np.ones(len(y_sy)),      # sintético → peso base
        np.full(len(y_re), 3.0), # real → peso triple
    ])
    w_c = w_c * domain

    sc  = StandardScaler()
    Xs  = sc.fit_transform(X_c)
    lr  = LogisticRegression(
        max_iter=2000,
        solver="saga",           # saga soporta sample_weight correctamente
        C=0.5,                   # regularización más fuerte para generalizar
        class_weight="balanced", # doble seguro sobre el desbalance residual
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lr.fit(Xs, y_c, sample_weight=w_c)
    lr._scaler = sc
    print("    ✓ LR-MFCC-FT v2 entrenada")
    return lr


def _train_rf_ft_v2(X_sy, y_sy, X_re, y_re):
    """
    RF sobre dataset combinado con class_weight adaptativo.

    Correcciones:
      - Un ÚNICO forest sobre el dataset combinado (sin concatenación).
        La concatenación original generaba data leakage porque los árboles
        sintéticos habían visto distribuciones distintas → ROC-AUC=1.0 espurio.
      - class_weight="balanced_subsample": cada árbol corrige el desbalance
        de clases dentro de su bootstrap sample → más robusto que 'balanced'.
      - max_features="sqrt" reduce varianza en datasets con muchas muestras.
      - min_samples_leaf=5 evita hojas con una sola muestra sintética.
    """
    import pandas as pd
    X_c = np.vstack([X_sy, X_re])
    y_c = np.concatenate([y_sy, y_re])

    # Peso extra al dominio real
    w_c = compute_sample_weight("balanced", y_c)
    domain = np.concatenate([
        np.ones(len(y_sy)),
        np.full(len(y_re), 3.0),
    ])
    w_c = w_c * domain

    rf = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced_subsample",  # corrige desbalance por árbol
        max_features="sqrt",
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    rf.fit(X_c, y_c, sample_weight=w_c)
    print("    ✓ RF-MFCC-FT v2 entrenado")
    return rf


# ══════════════════════════════════════════════════════════════════
# WAV2VEC2 FINE-TUNING CORREGIDO
# ══════════════════════════════════════════════════════════════════

class _AudioDS(torch.utils.data.Dataset):
    def __init__(self, paths, labels, processor, max_sec=5):
        self.paths, self.labels = paths, labels
        self.proc = processor
        self.maxl = TARGET_SR * max_sec

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        y = load_audio(self.paths[i])
        if y is None or len(y) == 0:
            y = np.zeros(TARGET_SR, dtype=np.float32)
        if len(y) > self.maxl:
            y = y[:self.maxl]
        enc = self.proc(
            y, sampling_rate=TARGET_SR, return_tensors="pt",
            padding=True, max_length=self.maxl, truncation=True,
        )
        return {
            "input_values": enc.input_values.squeeze(0),
            "label": torch.tensor(self.labels[i], dtype=torch.long),
        }


def _collate(batch):
    ml = max(b["input_values"].shape[0] for b in batch)
    iv = torch.zeros(len(batch), ml)
    am = torch.zeros(len(batch), ml)
    lb = []
    for i, b in enumerate(batch):
        l = b["input_values"].shape[0]
        iv[i, :l] = b["input_values"]
        am[i, :l] = 1
        lb.append(b["label"])
    return {
        "input_values":   iv,
        "attention_mask": am,
        "labels":         torch.stack(lb),
    }


def _init_head_weights(model):
    """
    Inicializa la cabeza de clasificación con varianza reducida (std=0.02).

    CORRECCIÓN PRINCIPAL: cuando se carga Wav2Vec2ForSequenceClassification
    desde un checkpoint ASR, la cabeza nueva (projector + classifier) se
    inicializa con la varianza por defecto de PyTorch (~0.1–0.3), que junto
    con las activaciones grandes del encoder produce NaN en el primer backward.
    std=0.02 es el valor estándar de GPT-2 y funciona bien con transformers.
    """
    for module in [model.projector, model.classifier]:
        if hasattr(module, "weight"):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if hasattr(module, "bias") and module.bias is not None:
            nn.init.zeros_(module.bias)


def _get_linear_warmup_scheduler(optimizer, n_warmup_steps, n_total_steps):
    """Warmup lineal durante n_warmup_steps, luego constante."""
    def lr_lambda(step):
        if step < n_warmup_steps:
            return float(step) / max(1, n_warmup_steps)
        return 1.0
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def _run_epoch(model, loader, optimizer=None, scaler=None, scheduler=None):
    """
    Una época de entrenamiento o evaluación.
    Soporta mixed precision (scaler != None).
    """
    train = optimizer is not None
    model.train() if train else model.eval()
    tot_loss, preds, trues = 0.0, [], []
    use_amp = scaler is not None and DEVICE.type == "cuda"

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for b in loader:
            iv = b["input_values"].to(DEVICE)
            am = b["attention_mask"].to(DEVICE)
            lb = b["labels"].to(DEVICE)

            if use_amp:
                with autocast():
                    out  = model(input_values=iv, attention_mask=am, labels=lb)
                    loss = out.loss
            else:
                out  = model(input_values=iv, attention_mask=am, labels=lb)
                loss = out.loss

            if train:
                optimizer.zero_grad()
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            loss_val = loss.item()
            if not np.isnan(loss_val):          # ignorar NaN esporádicos
                tot_loss += loss_val

            preds.extend(out.logits.argmax(1).cpu().tolist())
            trues.extend(lb.cpu().tolist())

    f1 = f1_score(trues, preds, average="macro", zero_division=0)
    return tot_loss / max(len(loader), 1), f1, preds, trues


def _freeze_encoder(model, n_layers: int):
    """Congela las primeras n_layers capas del encoder."""
    for p in model.wav2vec2.feature_extractor.parameters():
        p.requires_grad = False
    for i, layer in enumerate(model.wav2vec2.encoder.layers):
        if i < n_layers:
            for p in layer.parameters():
                p.requires_grad = False
        else:
            for p in layer.parameters():
                p.requires_grad = True


def _mk_loader(df, labels, processor, batch_size, shuffle):
    nw = 2 if DEVICE.type == "cuda" else 0
    return torch.utils.data.DataLoader(
        _AudioDS(df["wav_audio_path"].tolist(), labels, processor),
        batch_size=batch_size, shuffle=shuffle,
        collate_fn=_collate, num_workers=nw,
        pin_memory=(DEVICE.type == "cuda"),
    )


def _finetune_wav2vec2_v2(
    synth_df,
    real_df,
    save_path: str = str(MODELS_DIR / "wav2vec2_ft_v2"),
    # Fase 1: solo cabeza, encoder completamente congelado
    p1_epochs:  int   = 5,
    p1_lr:      float = 1e-3,    # lr alta está bien porque encoder está congelado
    p1_batch:   int   = 16,
    p1_warmup:  float = 0.1,     # 10 % de los pasos como warmup
    p1_patience:int   = 2,
    # Fase 2: descongelar capas superiores progresivamente
    p2_epochs:  int   = 8,
    p2_lr:      float = 5e-5,    # lr baja para no destruir representaciones
    p2_batch:   int   = 16,
    p2_freeze:  int   = 8,       # congela las primeras 8 de 12 capas
    p2_warmup:  float = 0.1,
    p2_patience:int   = 3,
) -> dict:
    """
    Fine-tuning Wav2Vec2 en dos fases con correcciones al NaN loss.

    FASE 1 — Solo cabeza (encoder 100% congelado):
      El encoder ya tiene representaciones ricas del habla.
      La cabeza nueva (projector + classifier) necesita aprender
      desde cero SIN que los gradientes del encoder la rompan.
      Congelar el encoder en Fase 1 elimina el NaN porque:
        a) No hay gradientes fluyendo por capas con activaciones grandes.
        b) La cabeza converge rápido con lr=1e-3 sobre representaciones estables.

    FASE 2 — Descongelar capas superiores del encoder:
      Una vez la cabeza está estabilizada, se descongelan las
      últimas (12 - p2_freeze) capas del encoder para adaptarlas
      al dominio de disfluencias, con lr muy baja (5e-5) para no
      destruir las representaciones preentrenadas.

    Optimizaciones de velocidad:
      - Mixed precision (fp16) en GPU: ~2x más rápido
      - Encoder congelado en Fase 1: ~4x menos operaciones por batch
      - Submuestreo sintético fluido antes de construir el DataLoader
      - num_workers=2 para prefetch en GPU
    """
    print(f"\n    [Wav2Vec2-FT v2] Dispositivo: {DEVICE}")

    proc = Wav2Vec2Processor.from_pretrained(WAV2VEC_ID)

    # ─ Balancear sintético antes de construir loaders ─────────────
    synth_bal = _subsample_synthetic_fluent(synth_df, real_df, target_ratio=1.5)
    print(f"    Sintético balanceado: {len(synth_bal)} clips "
          f"({synth_bal['is_disfluent'].sum()} disfluentes)")

    y_s = synth_bal["is_disfluent"].values
    idx = np.arange(len(synth_bal))
    tr_s, va_s = train_test_split(
        idx, test_size=0.2, stratify=y_s, random_state=RANDOM_STATE
    )

    y_r   = real_df["is_disfluent"].values
    idx_r = np.arange(len(real_df))
    tr_r, tmp = train_test_split(
        idx_r, test_size=0.5, stratify=y_r, random_state=RANDOM_STATE
    )
    va_r, te_r = train_test_split(
        tmp, test_size=0.5, stratify=y_r[tmp], random_state=RANDOM_STATE
    )

    dl_s_tr = _mk_loader(synth_bal.iloc[tr_s], y_s[tr_s].tolist(), proc, p1_batch, True)
    dl_s_va = _mk_loader(synth_bal.iloc[va_s], y_s[va_s].tolist(), proc, p1_batch, False)
    dl_r_tr = _mk_loader(real_df.iloc[tr_r],   y_r[tr_r].tolist(), proc, p2_batch, True)
    dl_r_va = _mk_loader(real_df.iloc[va_r],   y_r[va_r].tolist(), proc, p2_batch, False)
    dl_r_te = _mk_loader(real_df.iloc[te_r],   y_r[te_r].tolist(), proc, p2_batch, False)

    print(f"    Sintético → train:{len(tr_s)}  val:{len(va_s)}")
    print(f"    Real      → train:{len(tr_r)}  val:{len(va_r)}  test:{len(te_r)}")

    # ─ Cargar modelo e inicializar cabeza correctamente ───────────
    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        WAV2VEC_ID, num_labels=2
    ).to(DEVICE)
    _init_head_weights(model)          # ← CORRECCIÓN CLAVE

    scaler_amp = GradScaler() if DEVICE.type == "cuda" else None

    # ════════════════════════════════════════════════════════════
    # FASE 1: Encoder 100% congelado, solo entrena la cabeza
    # ════════════════════════════════════════════════════════════
    print(f"\n    [F1] Cabeza sobre sintético ({p1_epochs} épocas, encoder congelado) …")
    _freeze_encoder(model, n_layers=12)         # congela TODAS las capas
    for p in model.wav2vec2.feature_projection.parameters():
        p.requires_grad = False                 # congela también la proyección

    # Solo parámetros de la cabeza son entrenables
    head_params = list(model.projector.parameters()) + list(model.classifier.parameters())
    opt1 = torch.optim.AdamW(head_params, lr=p1_lr, weight_decay=0.01)

    n_steps_p1 = len(dl_s_tr) * p1_epochs
    sched1 = _get_linear_warmup_scheduler(opt1, int(n_steps_p1 * p1_warmup), n_steps_p1)

    h1        = {"loss": [], "val_f1": []}
    best_f1_1 = 0.0
    pat_ct_1  = 0

    for ep in range(p1_epochs):
        loss, _, _, _ = _run_epoch(model, dl_s_tr, opt1, scaler_amp, sched1)
        _, vf1, _, _  = _run_epoch(model, dl_s_va)
        h1["loss"].append(loss); h1["val_f1"].append(vf1)
        flag = " ← mejor" if vf1 > best_f1_1 else ""
        if vf1 > best_f1_1:
            best_f1_1 = vf1; pat_ct_1 = 0
        else:
            pat_ct_1 += 1
        print(f"      Ep {ep+1}/{p1_epochs}  loss={loss:.4f}  val_f1={vf1:.4f}{flag}")
        if pat_ct_1 >= p1_patience:
            print(f"      Early stopping Fase 1 en época {ep+1}"); break

    # ════════════════════════════════════════════════════════════
    # FASE 2: Descongelar capas superiores, fine-tune con reales
    # ════════════════════════════════════════════════════════════
    print(f"\n    [F2] Fine-tuning real ({p2_epochs} épocas, "
          f"primeras {p2_freeze} capas congeladas) …")
    _freeze_encoder(model, n_layers=p2_freeze)

    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"      Parámetros entrenables: "
          f"{sum(p.numel() for p in trainable):,} / "
          f"{sum(p.numel() for p in model.parameters()):,}")

    opt2 = torch.optim.AdamW(trainable, lr=p2_lr, weight_decay=0.01)

    n_steps_p2 = len(dl_r_tr) * p2_epochs
    sched2     = _get_linear_warmup_scheduler(opt2, int(n_steps_p2 * p2_warmup), n_steps_p2)

    h2         = {"loss": [], "val_f1": []}
    best_f1_2  = 0.0
    best_state = None
    pat_ct_2   = 0

    for ep in range(p2_epochs):
        loss, _, _, _ = _run_epoch(model, dl_r_tr, opt2, scaler_amp, sched2)
        _, vf1, _, _  = _run_epoch(model, dl_r_va)
        h2["loss"].append(loss); h2["val_f1"].append(vf1)
        flag = ""
        if vf1 > best_f1_2:
            best_f1_2  = vf1
            best_state = copy.deepcopy(model.state_dict())
            pat_ct_2   = 0
            flag       = " ← mejor"
        else:
            pat_ct_2 += 1
        print(f"      Ep {ep+1}/{p2_epochs}  loss={loss:.4f}  val_f1={vf1:.4f}{flag}")
        if pat_ct_2 >= p2_patience:
            print(f"      Early stopping Fase 2 en época {ep+1}"); break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.save_pretrained(save_path)
    proc.save_pretrained(save_path)
    print(f"\n    Modelo guardado: {save_path}")

    # ─ Evaluación en test real ────────────────────────────────────
    _, f1_te, preds, trues = _run_epoch(model, dl_r_te)
    print(f"\n    [Eval] Test Macro F1 = {f1_te:.4f}")
    print(classification_report(
        trues, preds, target_names=["Fluido", "Disfluente"], zero_division=0
    ))

    _plot_ft_curves(h1, h2)
    return {"test_macro_f1": f1_te, "h1": h1, "h2": h2}


def _plot_ft_curves(h1, h2):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, h, title in zip(
        axes,
        [h1, h2],
        ["Fase 1 — Cabeza sobre sintético (encoder congelado)",
         "Fase 2 — Fine-tuning real (capas superiores)"],
    ):
        ep = range(1, len(h["loss"]) + 1)
        ax.plot(ep, h["loss"], "b-o", label="Train Loss")
        ax2 = ax.twinx()
        ax2.plot(ep, h["val_f1"], "r-s", label="Val Macro F1")
        ax.set(xlabel="Época", ylabel="Loss", title=title)
        ax2.set_ylabel("Macro F1", color="r")
        lines  = ax.get_legend_handles_labels()[0]  + ax2.get_legend_handles_labels()[0]
        labels = ax.get_legend_handles_labels()[1]  + ax2.get_legend_handles_labels()[1]
        ax.legend(lines, labels, fontsize=9)
    plt.tight_layout()
    p = FIGS_DIR / "exp7_ft_curves_v2.png"
    plt.savefig(p, dpi=120); plt.close()
    print(f"    [Fig] {p.name}")


# ══════════════════════════════════════════════════════════════════
# FUNCIÓN PRINCIPAL — experiment_7 v2
# ══════════════════════════════════════════════════════════════════

def experiment_7(
    run_wav2vec_ft: bool = True,
    features: tuple = ("mfcc",),
    synth_target_ratio: float = 2.0,
) -> dict:
    """
    Experimento 7 v2 — Sintético → Fine-Tuning con reales (corregido).

    Cambios respecto a la versión original:
      - LR : ponderación adaptativa + class_weight='balanced' + solver saga
      - RF  : dataset combinado único (sin concatenación de forests)
              + class_weight='balanced_subsample' + min_samples_leaf=5
      - W2V : Fase 1 con encoder congelado (elimina NaN loss)
              + inicialización de cabeza con std=0.02
              + warmup lineal + mixed precision en GPU

    Args:
        run_wav2vec_ft     : ejecutar fine-tuning neuronal (requiere GPU
                             para tiempo razonable; ver exp7_patch.py para CPU)
        features           : features a usar para LR/RF ('mfcc', 'wav2vec')
        synth_target_ratio : ratio máximo fluido/disfluente tras submuestreo
                             sintético (default=2.0 → máx 2 fluidos por disfluente)

    Tiempo estimado (GPU T4):
        LR + RF MFCC    : ~5 min  (caché de features)
        Wav2Vec2-FT v2  : ~20-30 min (Fase 1: ~8 min, Fase 2: ~12 min)
        Total           : ~25-35 min
    """
    print("\n" + "╔" + "═"*58 + "╗")
    print("║  EXPERIMENTO 7 v2 — Sintético → Fine-Tuning (corregido)" + " "*2 + "║")
    print("╚" + "═"*58 + "╝")

    real_df  = load_real_data()
    synth_df = discover_synthetic_clips()
    results  = {}

    # ── Submuestrear sintético fluido ANTES de extraer features ──
    synth_bal = _subsample_synthetic_fluent(synth_df, real_df, synth_target_ratio)
    n_dis = synth_bal["is_disfluent"].sum()
    n_flu = (synth_bal["is_disfluent"] == 0).sum()
    print(f"\n  Dataset combinado tras balance: "
          f"{len(real_df) + len(synth_bal)} clips "
          f"| Fluido sintético: {n_flu} | Disfluente sintético: {n_dis}")

    for feat in features:
        print(f"\n  ── {feat.upper()} ──")

        # Extraer features (usa caché si existe)
        X_sy, y_sy, _ = build_dataset(
            synth_bal, feature=feat, cache_tag=f"exp7_synth_bal_{feat}"
        )
        X_re, y_re, meta_re = build_dataset(
            real_df, feature=feat, cache_tag=f"exp7_real_{feat}"
        )
        splits_re = make_splits(X_re, y_re, meta_re)

        # ── LR corregida ──────────────────────────────────────────
        print(f"\n  Entrenando LR-{feat.upper()}-FT v2 …")
        lr = _train_lr_ft_v2(X_sy, y_sy, X_re, y_re, splits_re)
        name = f"LR-{feat.upper()}-FT"
        results[name] = evaluate_splits(lr, splits_re, name)
        save_model(lr, name)

        # ── RF corregido ──────────────────────────────────────────
        print(f"\n  Entrenando RF-{feat.upper()}-FT v2 …")
        rf = _train_rf_ft_v2(X_sy, y_sy, X_re, y_re)
        name = f"RF-{feat.upper()}-FT"
        results[name] = evaluate_splits(rf, splits_re, name)
        save_model(rf, name)

    # ── Wav2Vec2 fine-tuning corregido ────────────────────────────
    if run_wav2vec_ft:
        if DEVICE.type == "cpu":
            print("\n  ⚠  Wav2Vec2-FT sobre CPU puede tardar >1h.")
            print("     Considera usar finetune_wav2vec2_embed() de exp7_patch.py.")
        print("\n  ── WAV2VEC2 Fine-Tuning v2 (encoder congelado en F1) ──")
        ft = _finetune_wav2vec2_v2(synth_df, real_df)
        results["Wav2Vec2-FT"] = {"test": ft}

    return results

In [ ]:
def run_pipeline(
    experiments:    list  = None,
    features:       tuple = ("mfcc", "wav2vec"),
    run_wav2vec_ft: bool  = True,
    source:         str   = "both",
) -> dict:
    """
    Punto de entrada principal.

    Args:
        experiments    : Lista de enteros [1..7]. None = todos.
        features       : Tupla de features a usar ("mfcc", "wav2vec").
        run_wav2vec_ft : Activar fine-tuning neuronal en Exp 7.
        source         : "real" | "synthetic" | "both"
                         Controla de dónde vienen los datos en los
                         experimentos que no tienen fuente fija.

    Ejemplo de uso:
        # Solo baseline real con MFCC
        run_pipeline(experiments=[1], features=("mfcc",), source="real")

        # Todos los experimentos con ambas features
        run_pipeline()

        # Solo sintético, experimentos 2 y 3
        run_pipeline(experiments=[2,3], source="synthetic")
    """
    if experiments is None:
        experiments = list(range(1, 8))

    print("\n" + "╔"+"═"*58+"╗")
    print("║  PIPELINE DISFLUENCIAS v3                             ║")
    print(f"║  Experimentos : {str(experiments):<43}║")
    print(f"║  Features     : {str(features):<43}║")
    print(f"║  Fuente datos : {source:<43}║")
    print(f"║  Dispositivo  : {str(DEVICE):<43}║")
    print("╚"+"═"*58+"╝")

    all_results = {}

    if 1 in experiments:
        all_results["Exp1_Real"] = experiment_1(features)

    if 2 in experiments:
        all_results["Exp2_RealSynth"] = experiment_2(features)

    if 3 in experiments:
        all_results["Exp3_Controlled"] = experiment_3(features=features)

    if 4 in experiments:
        all_results["Exp4_Severity"] = experiment_4(features=features[:1])

    if 5 in experiments:
        all_results["Exp5_Intensity"] = experiment_5(features=features[:1])

    if 6 in experiments:
        all_results["Exp6_Diversity"] = experiment_6(features=features[:1])

    if 7 in experiments:
        all_results["Exp7_FineTune"] = experiment_7(
            run_wav2vec_ft=run_wav2vec_ft, features=features[:1]
        )

    # Comparación global
    print("\n[K] Generando comparación global …")
    plot_experiment_comparison(all_results)

    print("\n" + "╔"+"═"*58+"╗")
    print("║  PIPELINE COMPLETADO                                  ║")
    print(f"║  Figuras  → {str(FIGS_DIR)[:46]:<46}║")
    print(f"║  Modelos  → {str(MODELS_DIR)[:46]:<46}║")
    print("╚"+"═"*58+"╝")
    return all_results

In [ ]:
# ejecutar pipeline para todos los experimentos
run_pipeline(experiments=[4], features=("mfcc", "wav2vec"), source="both")


╔══════════════════════════════════════════════════════════╗
║  PIPELINE DISFLUENCIAS v3                             ║
║  Experimentos : [4]                                        ║
║  Features     : ('mfcc', 'wav2vec')                        ║
║  Fuente datos : both                                       ║
║  Dispositivo  : cuda                                       ║
╚══════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════╗
║  EXPERIMENTO 4 — Ablación por severidad                   ║
╚══════════════════════════════════════════════════════════╝

  → Severidad: leve  (intensidad fija=media)
    Filtrado calidad: 4705 → 4676 clips
    Total: 4676 clips | Disfluentes: 1382 (29.6%)
    Speakers: 5
    Total clips: 12177 | Disfluentes: 374 (3.1%)
    Speakers   : 5
    Tipos      : {'fluent': 11803, 'rep_pal': 275, 'bloq': 99}

[load_data] Fuente='both' → 16853 clips totales (1756 disfluentes)

  ── MFCC ──
    [Cache] Carga

{'Exp4_Severity': {'leve': {'LR-MFCC-exp4_leve': {'val': {'model': 'LR-MFCC-exp4_leve',
     'split': 'val',
     'accuracy': 0.8971408020140262,
     'macro_f1': 0.5970657384043211,
     'f1_fluido': 0.9447876447876448,
     'f1_disfluente': 0.24934383202099739,
     'precision': 0.5191256830601093,
     'recall': 0.16407599309153714,
     'roc_auc': np.float64(0.7927807811055898),
     'by_disfluency_type': {'Block': 0.183,
      'Interjection': 0.0,
      'Prolongation': 0.1935,
      'SoundRep': 0.1655,
      'WordRep': 0.0667,
      'bloq': 0.0,
      'fluent': 0.4955,
      'rep_pal': 0.0},
     'by_severidad': {'leve': 0.0},
     'by_intensidad': {'media': 0.0}},
    'test': {'model': 'LR-MFCC-exp4_leve',
     'split': 'test',
     'accuracy': 0.8906868033081625,
     'macro_f1': 0.5621107589035987,
     'f1_fluido': 0.9414258188824662,
     'f1_disfluente': 0.1827956989247312,
     'precision': 0.4146341463414634,
     'recall': 0.11724137931034483,
     'roc_auc': np.float64(0